# ClaimWise Insurance Prediction — 17. Leakage-Free 5-Fold Cross-Validation

## 1. Objective

Evaluate **every model** of the project with **5-fold cross-validation in which the preprocessing
runs inside each fold**. This is the rigorous complement to the single 80/20 split used by the
production notebooks (04–10) and to the 3-fold early-stopping CV used for the Hist Gradient
Boosting hyperparameters.

| Rule | How it is respected |
|---|---|
| 5 folds | `KFold(5, shuffle=True, random_state=42)` for regression, `KFold(5, shuffle=True, random_state=42)` for classification |
| preprocessing inside the fold | `ColumnTransformer` is part of a `Pipeline` that is **refitted on the training rows of every fold** — no encoding/scaling sees the validation rows |
| leakage-free classification target | each fold derives its high-loss threshold from its **own training rows only** |
| models | the 5 regression models + Logistic Regression and Decision Tree classifier |
| report | Fold 1–5 results plus mean and standard deviation |

Artifacts go to `Outputs/CrossValidation/`.

## 2. Import Libraries

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    """Walk upwards from the notebook folder until the ClaimWise project is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "Dataset").is_dir() and (candidate / "Src").is_dir():
            return candidate
    raise FileNotFoundError(f"ClaimWise project root not found above: {start}")


PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_DIR = PROJECT_ROOT / "Dataset"
SRC_DIR = PROJECT_ROOT / "Src"
MODELS_DIR = PROJECT_ROOT / "Models"
OUTPUTS_DIR = PROJECT_ROOT / "Outputs"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from academic_common import ensure_dirs, save_figure, style_axes

from academic_common import CV_DIR, raw_feature_frame, classification_metrics

ensure_dirs()
print("cross-validation output folder:", CV_DIR)

cross-validation output folder: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/CrossValidation


In [3]:
# Src/model_training_utils.py switches matplotlib to the non-interactive Agg backend so the
# command-line scripts can run without a display. Switch back to the inline backend here so
# that every figure below is rendered inside this notebook.
%matplotlib inline

## 3. Data — Raw Features, Preprocessing Deferred to the Folds

Cross-validation must start from the **raw** ClaimWise file (`01_raw_claimwise_50000.csv`), because
the already-preprocessed split files were encoded/scaled once on the whole dataset. Starting there
would leak information from the future validation rows into the features. Here the encoding and
scaling are learned *inside every fold* instead.

In [4]:
X_raw, y_raw = raw_feature_frame()
CAT_COLS = [c for c in X_raw.columns if c.startswith("cat")]
NUM_COLS = [c for c in X_raw.columns if c.startswith("cont")]

print("raw feature matrix:", X_raw.shape, "| target:", y_raw.shape)
print("categorical columns:", len(CAT_COLS), "| continuous columns:", len(NUM_COLS))
print("missing values:", int(X_raw.isna().sum().sum()))
print("target range:", f"{y_raw.min():.2f} .. {y_raw.max():.2f} | mean {y_raw.mean():.2f}")

raw feature matrix: (50000, 130) | target: (50000,)
categorical columns: 116 | continuous columns: 14


missing values: 0
target range: 10.00 .. 85923.56 | mean 3035.01


## 4. The Fold-Local Preprocessing Pipeline

One `ColumnTransformer` does exactly what the production pipeline does — ordinal-encode the
categorical columns, standardise the continuous ones — but it lives **inside** the `Pipeline`, so
in every fold it is fitted only on that fold's training rows and only *applied* to the validation
rows (`handle_unknown="use_encoded_value"` keeps unseen categories safe).

Models that need fully scaled inputs (Logistic Regression) get one extra `StandardScaler` after the
column transformer.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler


def make_pipeline(estimator, scale_all: bool = False) -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT_COLS),
            ("num", StandardScaler(), NUM_COLS),
        ],
        remainder="drop",
    )
    steps = [("preprocess", preprocessor)]
    if scale_all:
        steps.append(("scale", StandardScaler()))
    steps.append(("model", estimator))
    return Pipeline(steps)


demo = make_pipeline(object(), scale_all=True)
print(demo)
print("\ntransformer fitted inside Pipeline.fit() -> leakage-free by construction")

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['cat1', 'cat2', 'cat3',
                                                   'cat4', 'cat5', 'cat6',
                                                   'cat7', 'cat8', 'cat9',
                                                   'cat10', 'cat11', 'cat12',
                                                   'cat13', 'cat14', 'cat15',
                                                   'cat16', 'cat17', 'cat18',
                                                   'cat19', 'cat20', 'cat21',
                                                   'cat22', 'cat23', 'cat24',
                                                   'cat25', 'cat26', 'cat27',
                                  

## 5. Regression — 5 Folds

Models and hyperparameters are imported from the production sources (`Src/*_model.py`), so the CV
runs **the same estimators** that are deployed; only the evaluation protocol differs. The Hist
Gradient Boosting uses its early-stopping CV variant (`build_cv_model`).

In [6]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from linear_regression_model import build_model as build_linear
from decision_tree_regressor_model import build_model as build_dt_reg
from random_forest_regressor_model import build_model as build_rf
from gradient_boosting_regressor_model import build_model as build_gb
from hist_gradient_boosting_regressor_model import build_cv_model as build_hgb

REG_MODELS = [
    ("Linear Regression", build_linear(), False),
    ("Decision Tree Regressor", build_dt_reg(), False),
    ("Random Forest Regressor", build_rf(), False),
    ("Gradient Boosting Regressor", build_gb(), False),
    ("Hist Gradient Boosting Regressor", build_hgb(), True),
]
for name, est, scaled in REG_MODELS:
    print(f"{name:35s} extra scaling layer: {scaled} | {est.__class__.__name__}")

Linear Regression                   extra scaling layer: False | LinearRegression
Decision Tree Regressor             extra scaling layer: False | DecisionTreeRegressor
Random Forest Regressor             extra scaling layer: False | RandomForestRegressor
Gradient Boosting Regressor         extra scaling layer: False | GradientBoostingRegressor
Hist Gradient Boosting Regressor    extra scaling layer: True | HistGradientBoostingRegressor


In [7]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
rows = []

for name, estimator, scale_all in REG_MODELS:
    for fold, (train_idx, valid_idx) in enumerate(kfold.split(X_raw), start=1):
        pipe = make_pipeline(estimator, scale_all=scale_all)
        pipe.fit(X_raw.iloc[train_idx], y_raw.iloc[train_idx])
        preds = pipe.predict(X_raw.iloc[valid_idx])
        y_true = y_raw.iloc[valid_idx]

        rows.append({
            "model": name,
            "fold": fold,
            "MAE": mean_absolute_error(y_true, preds),
            "MSE": mean_squared_error(y_true, preds),
            "RMSE": float(np.sqrt(mean_squared_error(y_true, preds))),
            "R2": r2_score(y_true, preds),
        })
        print(f"{name:35s} fold {fold} | MAE {rows[-1]['MAE']:9.4f} | RMSE {rows[-1]['RMSE']:9.4f} | R2 {rows[-1]['R2']:.6f}")

reg_scores = pd.DataFrame(rows)
reg_scores.to_csv(CV_DIR / "regression_5fold_scores.csv", index=False)
print("\nSaved:", CV_DIR / "regression_5fold_scores.csv")

Linear Regression                   fold 1 | MAE 1333.5562 | RMSE 2050.5245 | R2 0.482614


Linear Regression                   fold 2 | MAE 1320.7931 | RMSE 2165.4708 | R2 0.471818


Linear Regression                   fold 3 | MAE 1351.5023 | RMSE 2093.3296 | R2 0.491562


Linear Regression                   fold 4 | MAE 1324.8196 | RMSE 2040.5713 | R2 0.482784


Linear Regression                   fold 5 | MAE 1308.6175 | RMSE 2117.4795 | R2 0.463722


...

Hist Gradient Boosting Regressor    fold 1 | MAE 1197.4330 | RMSE 1903.8660 | R2 0.553977


Hist Gradient Boosting Regressor    fold 2 | MAE 1200.8493 | RMSE 1988.8196 | R2 0.554478


Hist Gradient Boosting Regressor    fold 3 | MAE 1220.4730 | RMSE 1944.0762 | R2 0.561480


Hist Gradient Boosting Regressor    fold 4 | MAE 1206.5134 | RMSE 1907.7488 | R2 0.547925


Hist Gradient Boosting Regressor    fold 5 | MAE 1191.5752 | RMSE 1943.8014 | R2 0.548086

Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/CrossValidation/regression_5fold_scores.csv


In [8]:
agg = reg_scores.groupby("model")[["MAE", "MSE", "RMSE", "R2"]].agg(["mean", "std"])
reg_summary = pd.DataFrame({
    "model": agg.index,
    **{f"{metric}_{stat}": agg[(metric, stat)].values
       for metric in ["MAE", "MSE", "RMSE", "R2"] for stat in ["mean", "std"]},
})
reg_summary.to_csv(CV_DIR / "regression_5fold_summary.csv", index=False)

print("Regression 5-fold CV — mean (std) over folds")
for row in reg_summary.itertuples():
    print(f"{row.model:35s} MAE {row.MAE_mean:9.4f} ({row.MAE_std:7.4f}) | "
          f"RMSE {row.RMSE_mean:9.4f} ({row.RMSE_std:7.4f}) | "
          f"R2 {row.R2_mean:.6f} ({row.R2_std:.6f})")
print("\nSaved:", CV_DIR / "regression_5fold_summary.csv")

Regression 5-fold CV — mean (std) over folds
Decision Tree Regressor             MAE 1613.5846 (26.9911) | RMSE 2618.3877 (77.3312) | R2 0.183556 (0.044351)
Gradient Boosting Regressor         MAE 1269.7191 (13.7351) | RMSE 1978.5871 (41.7442) | R2 0.533926 (0.017902)
Hist Gradient Boosting Regressor    MAE 1203.3688 (10.9899) | RMSE 1937.6624 (34.3980) | R2 0.553189 (0.005585)
Linear Regression                   MAE 1327.8577 (15.9755) | RMSE 2093.4752 (50.9980) | R2 0.478500 (0.010828)
Random Forest Regressor             MAE 1260.3884 ( 5.0347) | RMSE 1978.8128 (28.2846) | R2 0.533885 (0.011586)

Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/CrossValidation/regression_5fold_summary.csv


In [9]:
fig, ax = plt.subplots(figsize=(10.5, 6))
for name, _, _ in REG_MODELS:
    part = reg_scores[reg_scores["model"] == name].sort_values("fold")
    ax.plot(part["fold"], part["R2"], marker="o", linewidth=2, label=name)

style_axes(ax, "Regression 5-fold cross-validation — R² per fold", "fold", "R²")
ax.set_xticks([1, 2, 3, 4, 5])
ax.legend(fontsize=9, loc="lower right")
path = save_figure(fig, CV_DIR / "regression_5fold_scores.png")
print("Saved:", path)

Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/CrossValidation/regression_5fold_scores.png


## 6. Classification — 5 Folds, Threshold Learned Inside the Fold

ClaimWise has no categorical target, so the classification extension uses the documented derived
target `loss_high`. For cross-validation the threshold must not be taken from data the fold has not
seen yet, therefore:

* **each fold computes `threshold = median(training losses of that fold)`** and labels both sides
  with it;
* the validation rows are labelled with the *same* fold threshold (they never define it);
* the thresholds themselves are printed and stored — nothing is chosen by hand.

`StratifiedKFold` is not usable here because the label vector changes per fold; a shuffled `KFold`
is used instead (a median split keeps both classes close to balanced in every fold — verified in the
output below).

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

CLASS_MODELS = [
    ("Logistic Regression", LogisticRegression(max_iter=1000, random_state=42), True),
    ("Decision Tree Classifier", DecisionTreeClassifier(max_depth=20, min_samples_leaf=2, random_state=42), False),
]
for name, est, scaled in CLASS_MODELS:
    print(f"{name:30s} extra scaling layer: {scaled} | {est.__class__.__name__}")

Logistic Regression            extra scaling layer: True | LogisticRegression
Decision Tree Classifier       extra scaling layer: False | DecisionTreeClassifier


In [11]:
from sklearn.metrics import roc_auc_score

cls_rows = []
for name, estimator, scale_all in CLASS_MODELS:
    for fold, (train_idx, valid_idx) in enumerate(kfold.split(X_raw), start=1):
        y_train_part = y_raw.iloc[train_idx]
        y_valid = y_raw.iloc[valid_idx]

        threshold = float(np.median(y_train_part.to_numpy()))       # training rows only
        y_tr = (y_train_part.to_numpy() >= threshold).astype(int)
        y_va = (y_valid.to_numpy() >= threshold).astype(int)

        pipe = make_pipeline(estimator, scale_all=scale_all)
        pipe.fit(X_raw.iloc[train_idx], y_tr)
        y_pred = pipe.predict(X_raw.iloc[valid_idx])
        y_prob = pipe.predict_proba(X_raw.iloc[valid_idx])[:, 1]

        metrics = classification_metrics(y_va, y_pred)
        cls_rows.append({
            "model": name,
            "fold": fold,
            "threshold": threshold,
            **metrics,
            "roc_auc": roc_auc_score(y_va, y_prob),
        })
        print(f"{name:30s} fold {fold} | thr {threshold:8.2f} | acc {metrics['accuracy']:.4f} | "
              f"F1 {metrics['f1']:.4f} | AUC {cls_rows[-1]['roc_auc']:.4f}")

cls_scores = pd.DataFrame(cls_rows)
cls_scores.to_csv(CV_DIR / "classification_5fold_scores.csv", index=False)
print("\nSaved:", CV_DIR / "classification_5fold_scores.csv")

Logistic Regression            fold 1 | thr  2118.86 | acc 0.7585 | F1 0.7449 | AUC 0.8385


Logistic Regression            fold 2 | thr  2115.96 | acc 0.7676 | F1 0.7562 | AUC 0.8494


Logistic Regression            fold 3 | thr  2106.31 | acc 0.7643 | F1 0.7588 | AUC 0.8449


Logistic Regression            fold 4 | thr  2113.84 | acc 0.7666 | F1 0.7563 | AUC 0.8434


Logistic Regression            fold 5 | thr  2122.90 | acc 0.7605 | F1 0.7453 | AUC 0.8421


Decision Tree Classifier       fold 1 | thr  2118.86 | acc 0.6882 | F1 0.6696 | AUC 0.6922


Decision Tree Classifier       fold 2 | thr  2115.96 | acc 0.6917 | F1 0.6758 | AUC 0.6904


Decision Tree Classifier       fold 3 | thr  2106.31 | acc 0.6880 | F1 0.6756 | AUC 0.6969


Decision Tree Classifier       fold 4 | thr  2113.84 | acc 0.6918 | F1 0.6792 | AUC 0.6979


Decision Tree Classifier       fold 5 | thr  2122.90 | acc 0.6913 | F1 0.6701 | AUC 0.6825

Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/CrossValidation/classification_5fold_scores.csv


In [12]:
agg = cls_scores.groupby("model")[["accuracy", "precision", "recall", "f1", "roc_auc"]].agg(["mean", "std"])
cls_summary = pd.DataFrame({
    "model": agg.index,
    **{f"{metric}_{stat}": agg[(metric, stat)].values
       for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"] for stat in ["mean", "std"]},
})
cls_summary.to_csv(CV_DIR / "classification_5fold_summary.csv", index=False)

print("Classification 5-fold CV — mean (std) over folds")
for row in cls_summary.itertuples():
    print(f"{row.model:30s} acc {row.accuracy_mean:.4f} ({row.accuracy_std:.4f}) | "
          f"precision {row.precision_mean:.4f} ({row.precision_std:.4f}) | "
          f"recall {row.recall_mean:.4f} ({row.recall_std:.4f}) | "
          f"F1 {row.f1_mean:.4f} ({row.f1_std:.4f}) | AUC {row.roc_auc_mean:.4f} ({row.roc_auc_std:.4f})")

thresholds = cls_scores[cls_scores["model"] == "Logistic Regression"][["fold", "threshold"]]
print("\nper-fold thresholds (training median):")
print(thresholds.to_string(index=False))
print("\nSaved:", CV_DIR / "classification_5fold_summary.csv")

Classification 5-fold CV — mean (std) over folds
Decision Tree Classifier       acc 0.6902 (0.0019) | precision 0.7111 (0.0062) | recall 0.6407 (0.0063) | F1 0.6741 (0.0041) | AUC 0.6920 (0.0062)
Logistic Regression            acc 0.7635 (0.0039) | precision 0.7896 (0.0071) | recall 0.7183 (0.0063) | F1 0.7523 (0.0066) | AUC 0.8437 (0.0040)

per-fold thresholds (training median):
 fold  threshold
    1   2118.860
    2   2115.960
    3   2106.305
    4   2113.835
    5   2122.900

Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/CrossValidation/classification_5fold_summary.csv


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))

for name, _, _ in CLASS_MODELS:
    part = cls_scores[cls_scores["model"] == name].sort_values("fold")
    axes[0].plot(part["fold"], part["accuracy"], marker="o", linewidth=2, label=name)
    axes[1].plot(part["fold"], part["f1"], marker="o", linewidth=2, label=name)

for ax, metric, title in ((axes[0], "accuracy", "Accuracy per fold"),
                          (axes[1], "f1", "F1 per fold")):
    style_axes(ax, f"Classification 5-fold CV — {title}", "fold", metric)
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.legend(fontsize=9)

fig.tight_layout()
path = save_figure(fig, CV_DIR / "classification_5fold_scores.png")
print("Saved:", path)

Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/CrossValidation/classification_5fold_scores.png


## 7. Conclusion

Actual results of this executed run (5 folds, shuffled with `random_state=42`, preprocessing fitted
inside every fold):

**Regression — mean (std) R² over folds**

| Model | MAE mean (std) | RMSE mean (std) | R² mean (std) |
|---|---:|---:|---:|
| Linear Regression | 1327.86 (15.98) | 2093.48 (51.00) | 0.478500 (0.010828) |
| Decision Tree Regressor | 1613.58 (26.99) | 2618.39 (77.33) | 0.183556 (0.044351) |
| Random Forest Regressor | 1260.39 (5.03) | 1978.81 (28.28) | 0.533885 (0.011586) |
| Gradient Boosting Regressor | 1269.72 (13.74) | 1978.59 (41.74) | 0.533926 (0.017902) |
| Hist Gradient Boosting Regressor | 1203.37 (10.99) | 1937.66 (34.40) | 0.553189 (0.005585) |

Best regressor by mean R²: **Hist Gradient Boosting Regressor** (R² **0.553189**, std 0.005585).

**Classification — mean (std) over folds**

| Model | Accuracy | Precision | Recall | F1 | ROC AUC |
|---|---:|---:|---:|---:|---:|
| Logistic Regression | 0.7635 (0.0039) | 0.7896 (0.0071) | 0.7183 (0.0063) | 0.7523 (0.0066) | 0.8437 (0.0040) |
| Decision Tree Classifier | 0.6902 (0.0019) | 0.7111 (0.0062) | 0.6407 (0.0063) | 0.6741 (0.0041) | 0.6920 (0.0062) |

Per-fold thresholds (training median, printed in the output): 2106.31 – 2122.90.

Why this matters: every number above comes from folds whose encoding/scaling/threshold learned
**only** from the fold's own training rows, so the scores are not inflated by leakage. The single
split results in notebooks 04–10 are unchanged; this notebook adds the mandatory robust estimate.

Artifacts: `Outputs/CrossValidation/regression_5fold_scores.csv`,
`regression_5fold_summary.csv`, `regression_5fold_scores.png`,
`classification_5fold_scores.csv`, `classification_5fold_summary.csv`,
`classification_5fold_scores.png`.